# Necessary Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
)
from sentence_transformers import SentenceTransformer

from tqdm import tqdm
from google.colab import drive

# Loading Dataset

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/classified_headline.csv')
print(f"Total: {len(df)} | Fox: {(df['source']=='fox').sum()} | NBC: {(df['source']=='nbc').sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    df['headline'].tolist(),
    df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label'],
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Total: 13866 | Fox: 7907 | NBC: 5959
Train: 11092 | Test: 2774
Device: cuda


# Base Model: Frozen Embeddings + Classifier

Use a pretrained RoBERTa-large as a **fixed** feature extractor. Mean-pool the token vectors to produce a single 1024-dim sentence embedding per headline, then train a classifier on top. The transformer weights are not updated and only the downstream classifier learns from the data.

In [5]:
tokenizer = AutoTokenizer.from_pretrained('roberta-large')
roberta = AutoModel.from_pretrained('roberta-large').to(device)
roberta.eval()


def embed(texts, batch_size=32):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                       max_length=64, return_tensors='pt').to(device)
        with torch.no_grad():
            out = roberta(**enc)
        # Mean-pool, masking padding tokens
        mask = enc['attention_mask'].unsqueeze(-1).float()
        summed = (out.last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        embs = (summed / counts).cpu().numpy()
        all_embs.append(embs)
    return np.vstack(all_embs)


X_train_emb = embed(X_train)
X_test_emb = embed(X_test)
print(f"Embedding shape: {X_train_emb.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 87/87 [00:03<00:00, 22.19it/s]

Embedding shape: (11092, 1024)


## Logistic Regression

A linear classifier on the embeddings. Tests whether the embedding space is already linearly separable for source attribution.

In [6]:
logreg = LogisticRegression(max_iter=1000, C=1.0)
logreg.fit(X_train_emb, y_train)

y_pred = logreg.predict(X_test_emb)
print(f"LogReg accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['NBC', 'FoxNews']))

LogReg accuracy: 0.8576
              precision    recall  f1-score   support

         NBC       0.86      0.80      0.83      1192
     FoxNews       0.86      0.90      0.88      1582

    accuracy                           0.86      2774
   macro avg       0.86      0.85      0.85      2774
weighted avg       0.86      0.86      0.86      2774



## MLP

A 4-layer dense network with BatchNorm, GELU, and dropout. Adds nonlinearity in case the decision boundary in embedding space is not linear.

In [7]:
class DenseMLP(nn.Module):
    def __init__(self, in_dim=1024, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 2),
        )
    def forward(self, x):
        return self.net(x)


Xtr = torch.tensor(X_train_emb, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test_emb, dtype=torch.float32)
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

mlp = DenseMLP(in_dim=X_train_emb.shape[1]).to(device)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
criterion = nn.CrossEntropyLoss()

best_acc, best_preds = 0.0, None
for epoch in range(15):
    mlp.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    scheduler.step()

    mlp.eval()
    with torch.no_grad():
        preds = mlp(Xte.to(device)).argmax(dim=1).cpu().numpy()
    acc = accuracy_score(y_test, preds)
    if acc > best_acc:
        best_acc, best_preds = acc, preds
    print(f"Epoch {epoch+1:2d}  loss={total_loss/len(Xtr):.4f}  test_acc={acc:.4f}  best={best_acc:.4f}")

print(f"\nBest MLP accuracy: {best_acc:.4f}")
print(classification_report(y_test, best_preds, target_names=['NBC', 'FoxNews']))

Epoch  1  loss=0.4319  test_acc=0.8554  best=0.8554
Epoch  2  loss=0.3161  test_acc=0.8544  best=0.8554
Epoch  3  loss=0.2718  test_acc=0.8335  best=0.8554
Epoch  4  loss=0.2430  test_acc=0.8648  best=0.8648
Epoch  5  loss=0.2049  test_acc=0.8756  best=0.8756
Epoch  6  loss=0.1684  test_acc=0.8410  best=0.8756
Epoch  7  loss=0.1532  test_acc=0.7210  best=0.8756
Epoch  8  loss=0.1216  test_acc=0.8630  best=0.8756
Epoch  9  loss=0.1002  test_acc=0.8504  best=0.8756
Epoch 10  loss=0.0825  test_acc=0.8709  best=0.8756
Epoch 11  loss=0.0652  test_acc=0.8771  best=0.8771
Epoch 12  loss=0.0535  test_acc=0.8745  best=0.8771
Epoch 13  loss=0.0442  test_acc=0.8789  best=0.8789
Epoch 14  loss=0.0428  test_acc=0.8800  best=0.8800
Epoch 15  loss=0.0376  test_acc=0.8821  best=0.8821

Best MLP accuracy: 0.8821
              precision    recall  f1-score   support

         NBC       0.86      0.87      0.86      1192
     FoxNews       0.90      0.89      0.90      1582

    accuracy                 

## Sentence-Transformer (all-roberta-large-v1)

The previous embeddings came from RoBERTa pretrained on raw language modeling and was not optimized to produce good sentence-level representations. Sentence-Transformers like `all-roberta-large-v1` are explicitly trained on sentence-similarity tasks via contrastive learning, so semantically similar sentences map to nearby vectors. We test whether this purpose-built sentence embedder gives better classification accuracy than mean-pooled RoBERTa.

In [8]:
st_model = SentenceTransformer('sentence-transformers/all-roberta-large-v1', device=device)
X_train_emb_st = st_model.encode(X_train, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
X_test_emb_st  = st_model.encode(X_test,  batch_size=32, show_progress_bar=True, convert_to_numpy=True)
print(f"Sentence-Transformer embedding shape: {X_train_emb_st.shape}")

# LogReg
logreg_st = LogisticRegression(max_iter=2000, C=1.0)
logreg_st.fit(X_train_emb_st, y_train)
print(f"\nST + LogReg: {accuracy_score(y_test, logreg_st.predict(X_test_emb_st)):.4f}")

# MLP — reuse the same DenseMLP class
Xtr = torch.tensor(X_train_emb_st, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test_emb_st, dtype=torch.float32)
train_loader_st = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

mlp_st = DenseMLP(in_dim=X_train_emb_st.shape[1]).to(device)
optimizer = torch.optim.AdamW(mlp_st.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
criterion = nn.CrossEntropyLoss()

best_acc, best_preds = 0.0, None
for epoch in range(30):
    mlp_st.train()
    for xb, yb in train_loader_st:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        criterion(mlp_st(xb), yb).backward()
        optimizer.step()
    scheduler.step()
    mlp_st.eval()
    with torch.no_grad():
        preds = mlp_st(Xte.to(device)).argmax(dim=1).cpu().numpy()
    acc = accuracy_score(y_test, preds)
    if acc > best_acc:
        best_acc, best_preds = acc, preds
    if (epoch + 1) % 5 == 0 or epoch == 29:
        print(f"Epoch {epoch+1:2d}  test_acc={acc:.4f}  best={best_acc:.4f}")

print(f"\nST + MLP best: {best_acc:.4f}")
print(classification_report(y_test, best_preds, target_names=['NBC', 'FoxNews']))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/347 [00:00<?, ?it/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Sentence-Transformer embedding shape: (11092, 1024)

ST + LogReg: 0.7805
Epoch  5  test_acc=0.8342  best=0.8342
Epoch 10  test_acc=0.8226  best=0.8378
Epoch 15  test_acc=0.8205  best=0.8378
Epoch 20  test_acc=0.8298  best=0.8378
Epoch 25  test_acc=0.8295  best=0.8378
Epoch 30  test_acc=0.8317  best=0.8378

ST + MLP best: 0.8378
              precision    recall  f1-score   support

         NBC       0.81      0.81      0.81      1192
     FoxNews       0.86      0.86      0.86      1582

    accuracy                           0.84      2774
   macro avg       0.83      0.83      0.83      2774
weighted avg       0.84      0.84      0.84      2774



## Limitations of Frozen Embeddings

We tried two frozen embedders: mean-pooled RoBERTa-large (general LM pretraining) and `all-roberta-large-v1` (a Sentence-Transformer trained for sentence similarity). Both plateau in the same range, and surprisingly the purpose-built sentence embedder doesn't improve over mean-pooled RoBERTa. The likely reason: sentence-similarity training pushes embeddings of *semantically similar* sentences together, which actively suppresses the *stylistic* differences required for source attribution. News headline are semantically similar, but stylistically distinguishable. No matter how good the frozen embedder, it can only preserve information that survived its own pretraining objective. This caps performance around 86%.

# Fine-tuning

Instead of freezing the transformer, we update its weights jointly with the classification head. Gradients flow through the entire model, so the encoder's internal representations adapt specifically to the source-attribution task. The layers that previously encoded "general meaning" now encode meaning relevant to telling Fox apart from NBC. This typically gains 3 to 5% over the frozen approach.

In [7]:
!pip install -q --upgrade transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 129.2 MB/s eta 0:00:00


In [5]:
class HeadlineDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx],
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {'accuracy': accuracy_score(labels, np.argmax(logits, axis=1))}

## RoBERTa-base

125M parameters. Standard fine-tuning parameters: learning rate 2e-5, 4 epochs, 10% warmup, cosine decay.

In [9]:
ft_tokenizer = AutoTokenizer.from_pretrained('FacebookAI/roberta-base')
train_ds = HeadlineDataset(X_train, y_train, ft_tokenizer)
test_ds  = HeadlineDataset(X_test,  y_test,  ft_tokenizer)

ft_model = AutoModelForSequenceClassification.from_pretrained(
    'FacebookAI/roberta-base', num_labels=2,
).to(device)

args = TrainingArguments(
    output_dir='./roberta-base-finetuned',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=ft_model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

# Save
state_dict = {f"backbone.{k}": v for k, v in trainer.model.state_dict().items()}
torch.save(state_dict, "model_roberta_base.pt")

preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)
print(f"\nRoBERTa-base accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['NBC', 'FoxNews']))

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


RoBERTa-base accuracy: 0.8915
              precision    recall  f1-score   support

         NBC       0.85      0.90      0.88      1192
     FoxNews       0.92      0.88      0.90      1582

    accuracy                           0.89      2774
   macro avg       0.89      0.89      0.89      2774
weighted avg       0.89      0.89      0.89      2774



## RoBERTa-large

Now we try a model with more parameters (355M). Two adjustments matter for larger models: lower learning rate (1e-5 — half of base) since pretrained weights are more sensitive to large updates, and fp16 mixed precision plus gradient accumulation to fit reasonable effective batch sizes in memory.

In [10]:
ft_tokenizer_l = AutoTokenizer.from_pretrained('FacebookAI/roberta-large')
train_ds_l = HeadlineDataset(X_train, y_train, ft_tokenizer_l)
test_ds_l  = HeadlineDataset(X_test,  y_test,  ft_tokenizer_l)

ft_model_l = AutoModelForSequenceClassification.from_pretrained(
    'FacebookAI/roberta-large', num_labels=2,
).to(device)

args_l = TrainingArguments(
    output_dir='./roberta-large-finetuned',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer_l = Trainer(
    model=ft_model_l, args=args_l,
    train_dataset=train_ds_l, eval_dataset=test_ds_l,
    compute_metrics=compute_metrics,
)
trainer_l.train()

state_dict = {f"backbone.{k}": v for k, v in trainer_l.model.state_dict().items()}
torch.save(state_dict, "model_roberta_large.pt")

preds = trainer_l.predict(test_ds_l)
y_pred = np.argmax(preds.predictions, axis=1)
print(f"\nRoBERTa-large accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['NBC', 'FoxNews']))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


RoBERTa-large accuracy: 0.8901
              precision    recall  f1-score   support

         NBC       0.84      0.92      0.88      1192
     FoxNews       0.93      0.87      0.90      1582

    accuracy                           0.89      2774
   macro avg       0.89      0.89      0.89      2774
weighted avg       0.89      0.89      0.89      2774



## DeBERTa-v3-large

A different architecture (435M params) separately encodes content and position relationships. Generally outperforms RoBERTa on classification benchmarks. Same hyperparameters as RoBERTa-large.

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

args = TrainingArguments(
    output_dir='./deberta-v3-large-finetuned',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

# Final evaluation
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)
print(f"\nFinal test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['NBC', 'FoxNews']))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.343100,0.348213,0.848594
2,0.226200,0.279939,0.883201
3,0.100300,0.331645,0.897621
4,0.035900,0.398795,0.897621



Final test accuracy: 0.8976
              precision    recall  f1-score   support

         NBC       0.88      0.88      0.88      1192
     FoxNews       0.91      0.91      0.91      1582

    accuracy                           0.90      2774
   macro avg       0.90      0.90      0.90      2774
weighted avg       0.90      0.90      0.90      2774



# Conclusion
Fine-tuning decisively beats frozen embeddings (~89% vs ~86%), confirming that adapting the encoder weights to the task matters more than raw embedding quality. Across the three fine-tuned architectures, results land within ~1% of each other. After fine-tune, choice of backbone matters less than expected, and the bottleneck shifts from model capacity to data.

**Possible improvements:**
- **A larger and more topically diverse dataset** would likely raise the ceiling. Many errors come from wire-style factual headlines that genuinely could be from either source — more examples would help the model learn finer stylistic distinctions.
- **OpenAI's `text-embedding-3-large`** would be a natural additional comparison and may have produced stronger frozen-embedding results, but it isn't usable for this project: the evaluation backend has no internet access, so calling the OpenAI API at inference time is impossible.